In [1]:
import tensorflow as tf
from tensorflow.keras.applications.resnet import ResNet50, preprocess_input
from tensorflow.keras import layers, models

# 1. Define configuration constants
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 10  # Change this to match your dataset's total classes

# 2. Load the pre-trained ResNet50 base network
# We exclude the top classification layer (include_top=False) to adapt it to our dataset
print("Loading ResNet50 with ImageNet weights...")
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
)

# 3. Freeze the core feature extractor layers
# This prevents our optimizer from modifying the learned ImageNet features during early training
base_model.trainable = False

# 4. Construct the custom classification head using the Functional API
inputs = layers.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))

# Pass inputs through the specific preprocess_input function required by ResNet
x = preprocess_input(inputs)

# Pass the preprocessed tensors to the base feature extractor
x = base_model(x, training=False)

# Pool spatial feature maps into a 1D vector per image
x = layers.GlobalAveragePooling2D()(x)

# Add a dense layer with Batch Normalization and Dropout for regularization
x = layers.Dense(256, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)

# Output classification layer
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

# Instantiate the final model pipeline
model = models.Model(inputs, outputs)

# 5. Compile the computational graph
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Print out the model topology summary
model.summary()


Loading ResNet50 with ImageNet weights...
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 224, 224)  │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 224, 224)  │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 224, 224)  │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 224, 224,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 224, 224,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 7, 7,      │ 23,587,712 │ add[0][0]         │
│ (Functional)        │ 2048)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 2048)      │          0 │ resnet50[0][0]    │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │    524,544 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256)       │      1,024 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 10)        │      2,570 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,115,850 (91.99 MB)

 Trainable params: 527,626 (2.01 MB)

 Non-trainable params: 23,588,224 (89.98 MB)

In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.resnet import ResNet50, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image
from tensorflow.keras import layers, models

# =====================================================================
# 1. ARCHITECTURE SETUP (From previous step)
# =====================================================================
IMAGE_SIZE = (224, 224)
NUM_CLASSES = 1000  # Standard ImageNet classes for demo prediction

# Load the model with top layers included to test a real out-of-the-box prediction
print("Loading standard ResNet50 with classification head...")
model = ResNet50(weights='imagenet', include_top=True)


# =====================================================================
# 2. IMAGE LOADING & PREPROCESSING FUNCTION
# =====================================================================
def preprocess_local_image(image_path, target_size=IMAGE_SIZE):
    """
    Loads a local image file, scales it to the required size,
    and applies ResNet-specific normalization.
    """
    print(f"Loading image from path: {image_path}")
    # Load image and resize to 224x224
    img = image.load_img(image_path, target_size=target_size)

    # Convert PIL Image object to a numpy array (Shape: 224, 224, 3)
    img_array = image.img_to_array(img)

    # Expand dimensions to create a batch of 1 image (Shape: 1, 224, 224, 3)
    # Keras models expect batches of inputs, even for single items
    img_batch = np.expand_dims(img_array, axis=0)

    # Apply ResNet specific preprocessing (RGB to BGR conversion, zero-centering)
    preprocessed_img = preprocess_input(img_batch)

    return preprocessed_img


# =====================================================================
# 3. PREDICTION & POST-PROCESSING EXECUTION
# =====================================================================
def make_prediction(image_path):
    # Prepare the image tensor
    processed_data = preprocess_local_image(image_path)

    # Perform forward pass computation
    print("Running tensor inference through network...")
    predictions = model.predict(processed_data)

    # Decode predictions from probabilities back to human-readable labels
    # decode_predictions returns a list of tuples: (class_id, class_label, probability)
    # top=3 extracts the 3 highest confidence classifications
    decoded_results = decode_predictions(predictions, top=3)[0]

    print("\n=== Top 3 Predictions ===")
    for rank, (imagenet_id, label, confidence) in enumerate(decoded_results, 1):
        print(f"{rank}. {label.replace('_', ' ').title()}: {confidence * 100:.2f}%")


# =====================================================================
# HOW TO RUN IN GOOGLE COLAB:
# =====================================================================
# 1. Upload any test image (e.g., 'dog.jpg') to your Colab workspace file system.
# 2. Uncomment and run the execution line below:

make_prediction('/content/Make the Most of Your Fourth of July With the___.jpg')


Loading standard ResNet50 with classification head...
Loading image from path: /content/Make the Most of Your Fourth of July With the___.jpg
Running tensor inference through network...
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step

=== Top 3 Predictions ===
1. Alp: 84.38%
2. Valley: 7.02%
3. Cliff: 4.50%
